# dreamreel booster

Runs dreamreel's graphics-card AI (Indian voices, acting from a sentence, 3D models from a picture, auto-rigging,
faces driven by the voice, restyling) on this free notebook, for dreamreel on your own computer.

1. **Runtime → Change runtime type → T4 GPU** (free), then **Runtime → Run all**.
2. The last cell prints an **address** and a **pass key**. Paste them into dreamreel studio (Toolbox → Booster),
   or run the `dreamreel boost connect ...` line it prints.
3. Keep this tab open while you make your film. Models download the first time each one is used.

Only someone with the pass key can use this booster. Free notebooks stop after some hours or when idle;
just run it again and reconnect. For the Indian AI voices, add a free Hugging Face token as a Colab secret
called `HF_TOKEN` (the key icon on the left), after accepting the terms on
[ai4bharat/indic-parler-tts](https://huggingface.co/ai4bharat/indic-parler-tts) and
[ai4bharat/IndicF5](https://huggingface.co/ai4bharat/IndicF5). (dreamreel can also send it with each job.)

In [ ]:
#@title Settings
REPO = "https://github.com/nithinram97/archify"  #@param {type:"string"}
BRANCH = "claude/slate-code-json-upload-n9rp3l"  #@param {type:"string"}
PREPARE = "voice.indic_parler"  #@param {type:"string"}
# PREPARE: skills to install right away (comma-separated), e.g. voice.indic_parler,motion.hy_motion
# Leave empty to install each one the first time dreamreel asks for it.

In [ ]:
#@title Check the graphics card
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(gpu or 'No graphics card! Runtime -> Change runtime type -> T4 GPU, then run again.')

In [ ]:
#@title Get the booster
import os, subprocess
if not os.path.isdir('dreamreel-src'):
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', '--branch', BRANCH, REPO, 'dreamreel-src'], check=True)
    subprocess.run(['git', '-C', 'dreamreel-src', 'sparse-checkout', 'set', 'dreamreel/dreamreel/booster'], check=True)
else:
    subprocess.run(['git', '-C', 'dreamreel-src', 'pull', '--ff-only'], check=False)
SERVER = os.path.abspath('dreamreel-src/dreamreel/dreamreel/booster/server.py')
print('booster:', SERVER)

In [ ]:
#@title Hugging Face token (for the Indian AI voices)
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN found in Colab secrets')
except Exception:
    print('No HF_TOKEN secret here: dreamreel will send yours with each job if you saved one there (Toolbox -> Keys).')

In [ ]:
#@title Start the booster and give it a public address
import os, re, secrets, subprocess, sys, time
KEY = secrets.token_urlsafe(18)
log = open('booster.log', 'w')
booster = subprocess.Popen([sys.executable, SERVER, '--port', '8780', '--host', '127.0.0.1', '--token', KEY,
                            '--prepare', PREPARE], stdout=log, stderr=subprocess.STDOUT)
if not os.path.exists('cloudflared'):
    subprocess.run(['wget', '-q', '-O', 'cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod('cloudflared', 0o755)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8780', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        URL = m.group(0)
        break
time.sleep(3)
print('Address :', URL)
print('Pass key:', KEY)
print()
print('In dreamreel studio: Toolbox -> Booster, paste both. Or on your computer run:')
print(f'  dreamreel boost connect {URL} {KEY}')

In [ ]:
#@title Keep running (and show what the booster is doing)
import time
seen = 0
while booster.poll() is None:
    with open('booster.log') as f:
        lines = f.read().splitlines()
    for line in lines[seen:]:
        print(line)
    seen = len(lines)
    time.sleep(10)
print('The booster stopped. Run the cell above again.')